# Turtle Breakout 09 — risk per trade 1%–100% on BTC: long only vs long + stop-and-reverse

Note: `discussions/Turtle_Breakout/011_claude_tb09_note.md`. Code: `src/data_engineering/modeling/turtle_sizing.py` (`trade_list`, `equity_from_trades`, `sweep_trades`).

**Descriptive only — no decision gate, not for implementation** (choosing $f$ in-sample on a variant that itself failed G1 is double overfitting).
Same BTC window and sizing as TB 08: $u = f\cdot \mathrm{HWM}/(2\,\mathrm{ATR})$ for every trade. The reverse short (TB 07 rules) is sized the same way with the ATR of its own day.
Sweeps P (125× cap, HWM), C3 (3× cap), EQ (current-equity sizing), each for both trade lists: 600 backtests.

Environment: `TB01_DATA_DIR` (default `BINANCE_UM_DIR\long`), `TB09_SYNTHETIC=1`.

In [ ]:
import os, sys, json, subprocess, tempfile, warnings, platform, math
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 150)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, turtle_sizing as ts, turtle_reverse as tr

SYNTHETIC = os.getenv('TB09_SYNTHETIC', '0') == '1'
P = tb.Params(); SP = ts.SParams(); RP = tr.RParams()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb09-')) / '09'; SP = ts.SParams(n_boot=500)
    b = tb.synthetic_5m(2400, seed=11, trend=0.01, start='2020-01-01'); f = tb.synthetic_funding(b, 0.0001)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '09'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
    b, _ = tf.load_um(DATA / 'BTCUSDT_um_1m.parquet')
    f = tb.funding_events(pd.read_parquet(DATA / 'basis' / 'funding' / 'BTCUSDT_funding.parquet'))
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_sizing.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
D = tb.prepare_asset('BTC', b, f, P)
START = b.index[0] + pd.Timedelta(days=30)
L0 = ts.trade_list(D, P, START, RP, with_reverse=False)
L1 = ts.trade_list(D, P, START, RP, with_reverse=True)
print('BTC', START, '->', b.index[-1], '| longs', len(L0), '| reverse shorts', int((L1.kind == 'reverse').sum()))
print('days in a position: long only', round(float(((L0.jx - L0.je) / 288).sum()), 1), '| long + reverse', round(float(((L1.jx - L1.je) / 288).sum()), 1),
      '| of', round((b.index[-1] - START).days, 1))

## 1. Sweeps (600 backtests) and check against TB 08

In [ ]:
TAB, CUR = {}, {}
for name, cap, sb in (('P', SP.max_lev_primary, 'hwm'), ('C3', SP.max_lev_c3, 'hwm'), ('EQ', SP.max_lev_primary, 'equity')):
    for v, L in (('long', L0), ('long_rev', L1)):
        TAB[(name, v)], CUR[(name, v)] = ts.sweep_trades(D, L, P, START, cap, sb)
    print(name, 'done')
if not SYNTHETIC:
    for name in ('P', 'C3', 'EQ'):
        ref = pd.read_csv(ROOT / 'notebooks' / 'Turtle_Breakout' / '08' / f'08_sweep_{name}.csv')
        dif = np.nanmax(np.abs(ref.final_equity.to_numpy() - TAB[(name, 'long')].final_equity.to_numpy()) / ref.final_equity.abs().clip(lower=1).to_numpy())
        print(f'{name}: long-only sweep vs TB 08, max relative difference in final equity = {dif:.2e}')

## 2. Side by side at selected risk levels

In [ ]:
sel = [0.01, 0.02, 0.03, 0.05, 0.07, 0.1, 0.12, 0.13, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.75, 1.0]
cols = ['cagr', 'sharpe', 'max_dd', 'calmar', 'ruined', 'worst_trade_pct_equity', 'max_gross_lev']
SIDE = {}
for name in ('P', 'C3', 'EQ'):
    a = TAB[(name, 'long')].set_index('f')[cols]; c = TAB[(name, 'long_rev')].set_index('f')[cols + ['pnl_reverse']]
    S = pd.concat({'long': a, 'long_rev': c}, axis=1)
    S = S[S.index.round(3).isin([round(x, 3) for x in sel])]
    SIDE[name] = S; print(name); display(S.round(3))

## 3. Summary per sweep

In [ ]:
rows = []
for (name, v), t in TAB.items():
    ok = t[~t.ruined]
    best = t.loc[t.log_growth_per_year.idxmax()]
    rows.append({'sweep': name, 'variant': v, 'cagr_at_2pct': float(t.set_index('f').cagr[0.02]), 'max_dd_at_2pct': float(t.set_index('f').max_dd[0.02]),
                 'sharpe_at_2pct': float(t.set_index('f').sharpe[0.02]),
                 'best_f': float(best.f), 'best_cagr': float(best.cagr), 'best_max_dd': float(best.max_dd),
                 'first_ruin_f': float(t.loc[t.ruined, 'f'].min()) if t.ruined.any() else np.nan,
                 'max_f_with_dd_above_-50pct': float(t.loc[t.max_dd > -0.5, 'f'].max()) if (t.max_dd > -0.5).any() else np.nan,
                 'cagr_at_that_f': float(t.loc[t.max_dd > -0.5].iloc[-1].cagr) if (t.max_dd > -0.5).any() else np.nan})
SUM = pd.DataFrame(rows); display(SUM.round(3))

## 4. Long + reverse minus long only, at the same f (P sweep; paired block bootstrap; reported, no multiplicity control)

In [ ]:
A = CUR[('P', 'long')]; Bc = CUR[('P', 'long_rev')]
LA = np.log(A).diff().dropna(); LB = np.log(Bc).diff().dropna()
idx = ts.block_index(len(LA), SP.block_days, SP.n_boot, SP.seed)
rows = []
for f_ in A.columns:
    dl = (LB[f_] - LA[f_]).to_numpy(); bs = dl[idx].mean(1) * 365.25
    rows.append({'f': f_, 'growth_diff_per_year': dl.mean() * 365.25, 'lo95': float(np.quantile(bs, .025)), 'hi95': float(np.quantile(bs, .975)),
                 'dd_long': float(TAB[('P', 'long')].set_index('f').max_dd[f_]), 'dd_long_rev': float(TAB[('P', 'long_rev')].set_index('f').max_dd[f_])})
PAIR = pd.DataFrame(rows); display(PAIR[PAIR.f.round(3).isin([round(x, 3) for x in sel])].round(4))

## 5. Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sty = {'long': dict(color='#1f5aa6'), 'long_rev': dict(color='#b33f40')}
for name, ls in (('P', '-'), ('C3', '--'), ('EQ', ':')):
    for v in ('long', 'long_rev'):
        t = TAB[(name, v)]
        axes[0, 0].plot(t.f * 100, t.log_growth_per_year.clip(lower=-1), ls=ls, label=f'{name} {v}', **sty[v])
        axes[0, 1].plot(t.f * 100, t.max_dd * 100, ls=ls, label=f'{name} {v}', **sty[v])
axes[0, 0].set_ylabel('log growth per year (clipped at -1)'); axes[0, 1].set_ylabel('max drawdown %')
for a in axes[0]:
    a.axvline(2, color='k', lw=0.7, ls=':'); a.set_xlabel('risk per trade f (%)'); a.grid(alpha=0.25); a.legend(frameon=False, fontsize=7, ncol=2)
for f_, al in ((0.01, 0.45), (0.02, 1.0), (0.05, 0.75), (0.1, 0.55)):
    for v in ('long', 'long_rev'):
        c = CUR[('P', v)][f_]
        axes[1, 0].plot(c.index, c, alpha=al, lw=1.2, label=f'{v} f={f_:.0%}', **sty[v])
        axes[1, 1].plot(c.index, (c / c.cummax() - 1) * 100, alpha=al, lw=1.0, label=f'{v} f={f_:.0%}', **sty[v])
axes[1, 0].set_yscale('log'); axes[1, 0].set_ylabel('equity (log), P sweep'); axes[1, 1].set_ylabel('drawdown %, P sweep')
for a in axes[1]:
    a.grid(alpha=0.25); a.legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, f_ in zip(axes, (0.02, 0.1)):
    for v in ('long', 'long_rev'):
        c = CUR[('P', v)][f_]; ax.plot(c.index, (c / c.cummax() - 1) * 100, lw=1.1, label=v, **sty[v])
    ax.set_title(f'Drawdown at f = {f_:.0%} (P sweep)', loc='left', fontsize=10); ax.grid(alpha=0.25); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
for (name, v), t in TAB.items():
    t.to_csv(OUT / f'09_sweep_{name}_{v}.csv', index=False)
SUM.to_csv(OUT / '09_summary.csv', index=False); PAIR.to_csv(OUT / '09_pair_P.csv', index=False)
pd.concat(SIDE, axis=0).to_csv(OUT / '09_side_by_side.csv')
pd.concat({v: CUR[('P', v)][[0.01, 0.02, 0.05, 0.1]] for v in ('long', 'long_rev')}, axis=1).to_csv(OUT / '09_curves_P_selected.csv')
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '09_manifest.json').write_text(json.dumps({'version': ts.VERSION, 'tb01_version': tb.VERSION, 'tb07_version': tr.VERSION, 'git': git,
    'python': platform.python_version(), 'created_utc': pd.Timestamp.now('UTC').isoformat(), 'start': str(START),
    'longs': int(len(L0)), 'reverse_shorts': int((L1.kind == 'reverse').sum()), 'synthetic': SYNTHETIC, 'descriptive_only': True}, indent=1))
print('saved to', OUT)